In [1]:
import pandas as pd
import numpy as np

In [2]:
# 1) Carga de metadatos de las canciones (Music Info):
df_music = pd.read_csv('../data/raw/Music Info.csv')

# 2) Carga optimizada del historial de escuchas (Listening History): 
dtype_dict = {
    'track_id': 'string',
    'user_id': 'string',
    'playcount': 'uint32' 
}
df_history = pd.read_csv('../data/raw/User Listening History.csv', dtype=dtype_dict)

In [3]:
# 3) Filtrado k-core: 

def filter_k_core(df, user_col, item_col, k=10):
    """
    Filtra el DataFrame para mantener solo usuarios e ítems con al menos 'k' interacciones.
    """
    print(f"Dimensiones originales: {df.shape}")
    done = False
    
    while not done:
        starting_shape = df.shape[0]
        
        # Filtrar usuarios con menos de k escuchas:
        user_counts = df[user_col].value_counts()
        valid_users = user_counts[user_counts >= k].index
        df = df[df[user_col].isin(valid_users)]
        
        # Filtrar canciones con menos de k escuchas:
        item_counts = df[item_col].value_counts()
        valid_items = item_counts[item_counts >= k].index
        df = df[df[item_col].isin(valid_items)]
        
        # Si el tamaño no ha cambiado en esta iteración, hemos convergido: 
        if df.shape[0] == starting_shape:
            done = True
            
    print(f"Dimensiones tras filtrado {k}-core: {df.shape}")
    return df

# Aplicamos un filtrado de al menos 10 interacciones por usuario y canción: 
df_history_core = filter_k_core(df_history, 'user_id', 'track_id', k=10)

Dimensiones originales: (9711301, 3)
Dimensiones tras filtrado 10-core: (7027515, 3)


In [4]:
# 4) División en entrenamiento y test de datos: 

def split_implicit_data(df, user_col, test_size=0.2, random_state=42):
    """
    Divide los datos asegurando que cada usuario tenga interacciones tanto en train como en test.
    Requiere que los usuarios hayan pasado previamente por un filtro k-core.
    """
    print("Dividiendo datos en entrenamiento y prueba...")
    
    # Obtener índices de test muestreando del 20% de cada grupo:
    test_mask = df.groupby(user_col, group_keys=False).apply(
        lambda x: x.sample(frac=test_size, random_state=random_state)
    ).index
    
    df_test = df.loc[test_mask]
    df_train = df.drop(test_mask)
    
    print(f"Train shape: {df_train.shape} | Test shape: {df_test.shape}")
    return df_train, df_test

# Ejecutamos la división sobre nuestro dataset filtrado:
df_train, df_test = split_implicit_data(df_history_core, 'user_id')

Dividiendo datos en entrenamiento y prueba...
Train shape: (5627321, 3) | Test shape: (1400194, 3)
